# Smoke test Search-R1 sur Kaggle

**Avant de lancer** (panneau de droite, Settings) :
1. Accelerator : **GPU T4 x2** (ou P100).
2. Internet : **On** (compte vérifié par téléphone).
3. Dépôt privé : Add-ons > Secrets, créer `GITHUB_TOKEN` (jeton GitHub avec accès en lecture au dépôt). Dépôt public : rien à faire.

Puis **Run All**. Le test réussit si la dernière cellule affiche `SMOKE TEST RÉUSSI` (chaque trajectoire finit par `<answer>` ou par une recherche au dernier tour ; le taux de réponse est une mesure).
Durée attendue : quelques dizaines de minutes, surtout le téléchargement du modèle (3B, environ 6 Go).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess
from pathlib import Path

REPO = "vqlentinn/memoire-rl-search"
BRANCH = "corrections-smoke-test"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    token = None
    url = f"https://github.com/{REPO}.git"  # fonctionne si le dépôt est public

if not Path("memoire-rl-search").exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
    print(out)
    assert r.returncode == 0, "Clone échoué : vérifier le secret GITHUB_TOKEN ou rendre le dépôt public"
    # Retire le jeton de .git/config : /kaggle/working est publié comme sortie du notebook
    subprocess.run(["git", "-C", "memoire-rl-search", "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
%cd memoire-rl-search
!git log --oneline | head -3

## Dépendances

`vllm` peut entrer en conflit avec le PyTorch préinstallé sur Kaggle. Si l'installation échoue ou si la cellule
de test plante à l'import, **copiez l'erreur complète** : le repli prévu est Transformers (plus lent).

In [ ]:
!pip install -q vllm bm25s PyStemmer datasets huggingface_hub 2>&1 | tail -5
import vllm, torch
print("vllm", vllm.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
!VLLM_WORKER_MULTIPROC_METHOD=spawn python scripts/smoke_test.py --n 20 2>&1 | tee /kaggle/working/smoke_test.log

## Après le test

Copiez-collez dans la conversation **toute la sortie de la cellule précédente**, succès ou échec.
Les trajectoires sont dans `results/smoke_test.jsonl` (onglet Output du notebook après Save Version).